In [ ]:
# 1. Install necessary libraries and download language data
!pip install flickrapi pandas textblob
!python -m textblob.download_corpora

import flickrapi
import pandas as pd
import time
from textblob import TextBlob
from google.colab import files
from IPython.display import display

# 2. Enter your Flickr API credentials
API_KEY = 'c375be95249b84c36ff65ecb196a6eb6'
API_SECRET = 'b132f0b70bdf215c'

# Initialize Flickr API
flickr = flickrapi.FlickrAPI(API_KEY, API_SECRET, format='parsed-json')

# 3. Set search parameters for Castlefield, Manchester
castlefield_bbox = '-2.2625,53.4695,-2.2490,53.4785'
castlefield_keywords = [
    'castlefield', 'canal', 'canal basin', 'basin', 'waterside',
    'warehouse', 'wharf', 'marina', 'railway arches', 'arches',
    'viaduct', 'towpath', 'lock', 'bridge', 'walk', 'view',
    'urban landscape', 'historic building', 'waterfront', 'canalside'
]
tags_string = ','.join(castlefield_keywords)

print(f"Searching Flickr for the following keywords: {tags_string}")
print("Starting fetch and sentiment keyword extraction...")

try:
    data_for_csv = []
    current_page = 1
    max_pages = 20
    seen_ids = set() # 用于去重照片ID，防止API返回重复图片

    while current_page <= max_pages:
        print(f"Fetching data for page {current_page}...")

        photos = flickr.photos.search(
            tags=tags_string,
            tag_mode='any',
            bbox=castlefield_bbox,
            has_geo=1,
            extras='description', # 只需要描述和标题来做文本分析
            per_page=250,
            page=current_page
        )

        photo_list = photos['photos']['photo']

        if not photo_list:
            print("No more photos, fetching completed.")
            break

        for photo in photo_list:
            # 防止重复处理同一张照片
            photo_id = photo['id']
            if photo_id in seen_ids:
                continue
            seen_ids.add(photo_id)

            title = photo.get('title', '')
            desc_dict = photo.get('description', {})
            description = desc_dict.get('_content', '') if isinstance(desc_dict, dict) else ''

            # 合并标题和描述
            combined_text = f"{title} {description}".strip()

            if combined_text:
                blob = TextBlob(combined_text)
                overall_polarity = blob.sentiment.polarity

                # 核心过滤：只保留正负情绪，排除绝对中性 (0.0)
                if overall_polarity != 0.0:
                    sentiment_words = []

                    # 遍历句子中的每个词，找出带有情感分数的词
                    for word in blob.words:
                        word_polarity = TextBlob(word).sentiment.polarity
                        if word_polarity != 0.0:
                            # 转换为小写并存入列表
                            sentiment_words.append(word.lower())

                    # 单词去重（防止同一个情感词在同一段话里被列出多次）
                    sentiment_words = list(set(sentiment_words))

                    # 只有当确实提取到了情感关键词时，才加入最终数据
                    if sentiment_words:
                        keywords_string = ', '.join(sentiment_words)
                        data_for_csv.append({
                            'polarity': overall_polarity,
                            'keywords': keywords_string
                        })

        total_pages = int(photos['photos']['pages'])
        if current_page >= total_pages:
            print(f"Reached the last page ({total_pages}), fetching completed.")
            break

        current_page += 1
        time.sleep(1) # 遵守 API 调用频率限制

    print(f"\nSuccess! Extracted {len(data_for_csv)} raw records with positive or negative sentiment.")

    # 4. Generate CSV, Remove Duplicates, and Export
    if len(data_for_csv) > 0:
        df = pd.DataFrame(data_for_csv)

        # --- 数据清洗：删除极性和关键词完全一样的重复行 ---
        original_count = len(df)
        df = df.drop_duplicates(ignore_index=True)
        new_count = len(df)

        print(f"Data cleaning: Removed {original_count - new_count} duplicate rows.")
        print(f"Final unique records to export: {new_count}")
        # ---------------------------------------------------

        csv_filename = 'castlefield_sentiment_keywords.csv'
        df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

        print("Preparing to download the CSV file to your computer...")
        files.download(csv_filename)

        print("\nPreview of your clean, unique CSV:")
        display(df.head(10)) # 在 Colab 界面预览前10行
    else:
        print("No photos found matching criteria with positive or negative sentiment.")

except flickrapi.exceptions.FlickrError as e:
    print(f"API call error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
Finished.
Searching Flickr for the following keywords: castlefield,canal,canal basin,basin,waterside,warehouse,wharf,marina,railway arches,arches,viaduct,towpath,lock,bridge,walk,view,urban landscape,historic buil

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Preview of your clean, unique CSV:


,polarity,keywords
0,-0.125000,past
1,0.083571,"behind, real, high, first, originally"
2,0.500000,lovely
3,0.125000,first
4,0.160000,high
5,0.183333,"great, black, professional"
6,0.066518,"behind, large, long, closed, great, nearly, he..."
7,0.062605,"behind, large, long, closed, great, nearly, he..."
8,0.074370,"behind, large, long, closed, great, nearly, he..."
9,0.187500,originally
